# Emotion Recognition - Mini-Xception Training (Colab + GPU)

Bu notebook, GPU uzerinde **sadece Mini-Xception** modelini egitir ve test sonuclarini gosterir.

**Workflow:** GitHub Clone -> Kaggle Dataset -> GPU Training -> Test Sonuclari

**Onemli:** Runtime > Change runtime type > T4 GPU sectiginizden emin olun!

## 1. GPU Kontrolu

In [1]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")

Tue Mar  3 18:02:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. GitHub Repo Klonlama

In [2]:
import os

REPO_URL = "https://github.com/aysenurhepguven0/Emotion-Aware-Adaptive-Virtual-Interaction-System.git"
REPO_NAME = "Emotion-Aware-Adaptive-Virtual-Interaction-System"

%cd /content
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f"{REPO_NAME} zaten mevcut, pull yapiliyor...")
    !cd {REPO_NAME} && git pull

%cd /content/{REPO_NAME}
print(f"[OK] Calisma dizini: {os.getcwd()}")

/content
Cloning into 'Emotion-Aware-Adaptive-Virtual-Interaction-System'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 65 (delta 25), reused 60 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 286.57 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/Emotion-Aware-Adaptive-Virtual-Interaction-System
[OK] Calisma dizini: /content/Emotion-Aware-Adaptive-Virtual-Interaction-System


## 3. Bagimliliklari Yukleme

In [3]:
!pip install -q hsemotion --no-deps
!pip install -q kagglehub

  Preparing metadata (setup.py) ... done


## 4. Dataset Indirme (KaggleHub)

KaggleHub ile FER2013 ve FERPlus datasetlerini indirir.

Kaggle hesabinizla otomatik authenticate olur.

In [4]:
import kagglehub
import shutil
import os

# ----- FER2013 -----
print("FER2013 indiriliyor...")
fer_path = kagglehub.dataset_download("msambare/fer2013")
print(f"FER2013 path: {fer_path}")

# data/fer2013 klasorune kopyala
os.makedirs("data/fer2013", exist_ok=True)
!cp -r {fer_path}/* data/fer2013/
print("[OK] FER2013 yuklendi")
!ls data/fer2013/

FER2013 indiriliyor...
Using Colab cache for faster access to the 'fer2013' dataset.
FER2013 path: /kaggle/input/fer2013
[OK] FER2013 yuklendi
test  train


In [5]:
# ----- FER+ (FERPlus) -----
print("FERPlus indiriliyor...")
ferplus_path = kagglehub.dataset_download("arnabkumarroy02/ferplus")
print(f"FERPlus path: {ferplus_path}")

# data/ferplus klasorune kopyala
os.makedirs("data/ferplus", exist_ok=True)
!cp -r {ferplus_path}/* data/ferplus/
print("[OK] FERPlus yuklendi")
!ls data/ferplus/

FERPlus indiriliyor...
Using Colab cache for faster access to the 'ferplus' dataset.
FERPlus path: /kaggle/input/ferplus
[OK] FERPlus yuklendi
test  train  validation


In [6]:
# Dataset yapisini kontrol et
import os

for ds_name in ['fer2013', 'ferplus']:
    ds_path = f'data/{ds_name}'
    if os.path.exists(ds_path):
        print(f"\n{ds_name}/")
        for split in sorted(os.listdir(ds_path)):
            split_path = os.path.join(ds_path, split)
            if os.path.isdir(split_path):
                classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
                total = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
                print(f"  {split}: {total:,} images ({len(classes)} classes)")
    else:
        print(f"[ERROR] {ds_name} bulunamadi!")


fer2013/
  test: 7,178 images (7 classes)
  train: 28,709 images (7 classes)

ferplus/
  test: 3,573 images (8 classes)
  train: 66,379 images (8 classes)
  validation: 8,341 images (8 classes)


## 5. Config Ayarlari (GPU icin)

In [7]:
import torch
import config

if torch.cuda.is_available():
    config.EPOCHS = 30
    config.MODEL_CONFIGS["mini_xception"]["batch_size"] = 64
    print(f"[GPU] {config.EPOCHS} epoch, batch_size=64")
else:
    config.EPOCHS = 10
    print(f"[CPU] {config.EPOCHS} epoch")

print(f"Device: {config.DEVICE}")


[GPU] 30 epoch, batch_size=64
Device: cuda


## 6. Mini-Xception Model Egitimi

Secilen dataset uzerinde sadece Mini-Xception modeli egitir.

In [8]:
# ============================================
# DATASET SECIN
# ============================================
DATASET = "ferplus"  # "fer2013", "ferplus", "rafdb", "ckplus"

!python main.py --mode train --model mini_xception --dataset {DATASET}



  Emotion Recognition System
  Model: mini_xception | Dataset: ferplus

  Project dir:  /content/Emotion-Aware-Adaptive-Virtual-Interaction-System
  Device:       cuda
  Python:       3.12.12

  STEP 2: MODEL TRAINING (mini_xception)

[MODEL] Mini-Xception
  Total parameters:     347,255
  Trainable parameters: 347,255
  Number of classes:    7
  Input channels:       1

  Loading FER+ Dataset (img=48x48, ch=1)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/train: 58379 images loaded (7 classes)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/validation: 7341 images loaded (7 classes)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/test: 3543 images loaded (7 classes)

[INFO] FER+ Dataset sizes:
  Train:      58,379 samples
  Validation: 7,341 samples
  Test:       3,543 samples
  Batch:      64
  Device:     cuda

[INFO] Class weights:
  Angry       : 1.0425  (8,000 samples)
  Disgust     : 

## 7. Test Sonuclari (Mini-Xception)

In [9]:
import os
import config
from evaluate import evaluate_model
from models.mini_xception import get_model

MODEL_PATH = config.BEST_MODEL_PATHS["mini_xception"]
if not os.path.exists(MODEL_PATH):
    print("[ERROR] Mini-Xception modeli bulunamadi. Once egitim yapin.")
else:
    print(f"[INFO] Model yukleniyor: {MODEL_PATH}")
    model = get_model(pretrained_path=MODEL_PATH)
    results = evaluate_model(model=model, dataset_name=DATASET, split_name=f"{DATASET} Test")
    y_true = results["y_true"]
    y_pred = results["y_pred"]

    neutral_label = next(
        k for k, v in config.EMOTION_LABELS.items() if v.lower() == "neutral"
    )
    neutral_mask = y_true == neutral_label
    if neutral_mask.any():
        neutral_acc = (y_pred[neutral_mask] == neutral_label).mean() * 100
        print(f"Neutral class accuracy: {neutral_acc:.2f}% (n={neutral_mask.sum()})")
    else:
        print("[WARNING] Neutral sinifi test setinde bulunamadi.")


[INFO] Model yukleniyor: /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/outputs/models/best_mini_xception.pth
[INFO] Loading pretrained model: /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/outputs/models/best_mini_xception.pth
[INFO] Pretrained model loaded successfully.

[MODEL] Mini-Xception
  Total parameters:     347,255
  Trainable parameters: 347,255
  Number of classes:    7
  Input channels:       1

  Loading FER+ Dataset (img=48x48, ch=1)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/train: 58379 images loaded (7 classes)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/validation: 7341 images loaded (7 classes)
[INFO] /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/data/ferplus/test: 3543 images loaded (7 classes)

[INFO] FER+ Dataset sizes:
  Train:      58,379 samples
  Validation: 7,341 samples
  Test:       3,543 samples
  Batch:      64
  Device:     cuda


  MODEL DEĞERLE

Değerlendirme: 100%|██████████| 56/56 [00:02<00:00, 22.27it/s]



--------------------------------------------------
  Genel Doğruluk: 69.38%
--------------------------------------------------

  Classification Report:
              precision    recall  f1-score   support

       Angry     0.5939    0.6087    0.6012       322
     Disgust     0.1250    0.6667    0.2105        21
        Fear     0.3680    0.4694    0.4126        98
       Happy     0.8605    0.8170    0.8382       929
         Sad     0.4455    0.5278    0.4832       449
    Surprise     0.8060    0.7111    0.7556       450
     Neutral     0.7605    0.6954    0.7265      1274

    accuracy                         0.6938      3543
   macro avg     0.5656    0.6423    0.5754      3543
weighted avg     0.7228    0.6938    0.7055      3543

[PLOT] Confusion matrix kaydedildi: /content/Emotion-Aware-Adaptive-Virtual-Interaction-System/outputs/plots/confusion_matrix.png

  Per-class Accuracy:
    Angry       : 60.9%
    Disgust     : 66.7%
    Fear        : 46.9%
    Happy       : 81.7%
